# 第 5 天 – 人工智能宣传册生成器

该笔记本为初创公司构建了人工智能驱动的宣传册生成器。

系统：

1. 从公司网站上抓取链接
2. 使用法学硕士来过滤与投资者相关的页面
3. 生成结构化的启动手册
4. （可选）将手册翻译成另一种语言
5. 流输出以获得交互式体验

所有输出在提交前都会被清除。

## 架构概述

该系统遵循模块化管道：

网站  
↓  
链接抓取  
↓  
LLM链接过滤  
↓  
宣传册生成  
↓  
可选翻译

每个阶段都作为单独的功能实现，以保持系统模块化且易于扩展。

## 安装依赖项

我们安装所需的库：

- 网页抓取
- HTML解析
- LLM API调用

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json
import os

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

# 加载环境变量
load_dotenv(find_dotenv())

# 初始化OpenAI客户端
client = OpenAI()

# 使用型号
MODEL = "gpt-5-nano"

## 用户输入

用户提供：

- 网站网址
- 可选翻译语言

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
site_url = "https://edwarddonner.com/"

target_language = "Spanish"   # None for English output

## 步骤 1 – 抓取网站链接

这个功能：

- 获取网页
- 提取所有锚标签
- 将相对 URL 转换为绝对 URL

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def scrape_links(base_url):

    response = requests.get(base_url)
    soup = BeautifulSoup(response.text, "html.parser")

    links = set()

    for tag in soup.find_all("a", href=True):

        href = tag["href"]
        absolute = urljoin(base_url, href)

        if absolute.startswith("http"):
            links.add(absolute)

    return list(links)

## 步骤 2 – 使用 LLM 过滤相关链接

许多网站链接对于投资者手册没有用处。

我们使用法学硕士仅保留相关链接，例如：

- 产品页面
- 技术概述
- 公司概况
- 案例研究

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
LINK_FILTER_SYSTEM_PROMPT = """
You are an AI assistant helping create an investor brochure.

From a list of website links, return ONLY links useful for understanding the company.

Include:
- product pages
- technology explanation
- company overview
- pricing
- solutions
- case studies

Exclude:
- privacy policy
- terms
- cookies
- login
- careers
- legal pages

Return JSON:

{
 "relevant_links":[
   {"url":"...","reason":"..."}
 ]
}
"""

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def filter_links(links):

    user_prompt = f"""
Analyze the following links and return only relevant ones.

Links:
{json.dumps(links, indent=2)}
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":LINK_FILTER_SYSTEM_PROMPT},
            {"role":"user","content":user_prompt}
        ]
    )

    return response.choices[0].message.content

## 步骤 3 – 生成投资者手册

使用过滤后的链接，模型生成结构化的手册
对于初创投资者。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
BROCHURE_SYSTEM_PROMPT = """
You are an expert startup analyst.

Generate a professional investor brochure.

Include sections:

Product Overview
Target Market
Business Model
Technology
Key Advantages

Use concise professional language.
"""

## 流输出

为了使笔记本具有交互性，响应会逐个令牌进行流式传输。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def stream_response(system_prompt, user_prompt):

    stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":system_prompt},
            {"role":"user","content":user_prompt}
        ],
        stream=True
    )

    output=""

    for chunk in stream:

        token = chunk.choices[0].delta.content

        if token:
            print(token, end="", flush=True)
            output += token

    print("\n")

    return output

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def generate_brochure(filtered_links):

    user_prompt = f"""
Generate an investor brochure using the following company links:

{filtered_links}
"""

    return stream_response(
        BROCHURE_SYSTEM_PROMPT,
        user_prompt
    )

## 步骤 4 – 翻译手册

生成的手册可以为国际投资者翻译。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
TRANSLATION_SYSTEM_PROMPT = """
You are a professional business translator.

Translate the brochure into the requested language.

Preserve formatting and tone.
"""

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def translate_brochure(text, language):

    user_prompt = f"""
Translate the following brochure into {language}:

{text}
"""

    return stream_response(
        TRANSLATION_SYSTEM_PROMPT,
        user_prompt
    )

## 步骤 5 – 运行完整管道

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
links = scrape_links(site_url)

filtered_links = filter_links(links)

brochure = generate_brochure(filtered_links)

if target_language:
    brochure = translate_brochure(brochure, target_language)
from IPython.display import Markdown, display
display(Markdown(brochure))

html_content = f"<html><body><pre>{brochure}</pre></body></html>"

with open("brochure_output.html", "w", encoding="utf-8") as f:
    f.write(html_content)

